# 01 — Fuentes, limpieza e integración de datos

**Proyecto:** *Del colegio a la universidad: inteligencia territorial para cerrar brechas
de acceso y permanencia en la educación superior de Antioquia.*
Concurso **Datos al Ecosistema 2026 — IA para Colombia** (nivel intermedio).

## Fuentes integradas (5 conjuntos, 4 de datos.gov.co)

| # | Conjunto | Fuente | Filas |
|---|---|---|---|
| 1 | Matriculados UdeA sedes regionales 2026-1 (`URABA 20261.xlsx`) | Universidad de Antioquia | 8.157 |
| 2 | Beneficiarios de programas de acompañamiento ([xk8x-i6kn](https://www.datos.gov.co/d/xk8x-i6kn)) | datos.gov.co — Gobernación de Antioquia | 13.778 |
| 3 | Población Antioquia censada 2018 ([evm3-92yw](https://www.datos.gov.co/d/evm3-92yw)) | datos.gov.co — Gobernación de Antioquia / DANE | 12.825 |
| 4 | Resultados únicos Saber 11, filtro Antioquia 2018+ ([kgxf-xxbe](https://www.datos.gov.co/d/kgxf-xxbe)) | datos.gov.co — ICFES | 297.962 |
| 5 | DIVIPOLA códigos de municipios ([gdxc-w37w](https://www.datos.gov.co/d/gdxc-w37w)) | datos.gov.co — DANE | 1.122 |

Los conjuntos 4 y 5 se descargan por la **API Socrata** con `src/acquire.py`
(el Saber 11 se filtra en el servidor con SoQL: solo colegios de Antioquia y periodos
2018-1 en adelante, 298 mil de 7,1 millones de filas).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

## Problemas de calidad detectados y tratados

1. **Mojibake por doble codificación** en Beneficiarios: `SOÃ‘ARES` → `SOÑARES`,
   `URABÃ` → `URABÁ` (se corrige con `ftfy`).
2. **Categorías inconsistentes**: género `F`/`FEMENINO`/`M`/`MASCULINO`;
   víctima del conflicto `0`/`NO`/`1`/`SI`; subregión `URABA`/`URABÁ`.
3. **Fechas en formatos mixtos** en URABA: celdas datetime y textos `13-APR-1993`.
4. **Códigos de municipio incompatibles**: URABA usa el código DANE *sin* el prefijo
   de departamento (`854` → `05854` Valdivia); se homologa todo a DANE de 5 dígitos.
5. **Centinela `9.99`** en `PROMEDIO_PROGRAMA` para estudiantes sin historia académica
   (se detecta y se excluye de la población del modelo predictivo).

### Evidencia del mojibake (antes → después)

In [2]:
import ftfy
raw = pd.read_excel("../data/raw/Beneficiarios.xlsx",
                    sheet_name="Beneficiarios_de_los_programas_",
                    usecols=["PROGRAMA", "SUBREGIÓN DE RESIDENCIA"])
ejemplos = raw[raw["PROGRAMA"].str.contains("Ã", na=False) |
               raw["SUBREGIÓN DE RESIDENCIA"].str.contains("Ã", na=False)]
antes = ejemplos.drop_duplicates().head(5)
despues = antes.map(ftfy.fix_text)
pd.concat({"antes": antes.reset_index(drop=True),
           "después": despues.reset_index(drop=True)}, axis=1)

antes                           después         
  SUBREGIÓN DE RESIDENCIA  PROGRAMA SUBREGIÓN DE RESIDENCIA PROGRAMA
0                SUROESTE  SOÃ‘ARES                SUROESTE  SOÑARES
1              BAJO CAUCA  SOÃ‘ARES              BAJO CAUCA  SOÑARES
2                   URABA  SOÃ‘ARES                   URABA  SOÑARES
3                NORDESTE  SOÃ‘ARES                NORDESTE  SOÑARES
4                 ORIENTE  SOÃ‘ARES                 ORIENTE  SOÑARES

### Evidencia de los formatos mixtos de fecha

In [3]:
fechas = pd.read_excel("../data/raw/URABA 20261.xlsx",
                       sheet_name="20261_RG_MAT", usecols=["FECH_NACE"])
fechas["tipo_de_dato"] = fechas["FECH_NACE"].map(lambda v: type(v).__name__)
print(fechas["tipo_de_dato"].value_counts())
fechas.groupby("tipo_de_dato").head(2)

tipo_de_dato
datetime    5449
str         2708
Name: count, dtype: int64


,FECH_NACE,tipo_de_dato
0,13-APR-1993,str
1,2001-09-19 00:00:00,datetime
2,1968-05-26 00:00:00,datetime
3,11-APR-1974,str


## Ejecución del pipeline de limpieza e integración

`src/clean.py` normaliza cada fuente y la guarda en parquet; `src/integrate.py`
homologa municipios contra DIVIPOLA (llave: código DANE de 5 dígitos, con auditoría
difusa de nombres vía `rapidfuzz`) y construye las dos matrices analíticas.

In [4]:
import clean
clean.main()

URABA: 8157 filas | fechas no parseadas: 0 | sin cod DANE vive: 1 | estrato faltante: 324


Beneficiarios: 13778 filas | género: {'F': np.int64(7187), 'M': np.int64(4274), 'ND': np.int64(2316), 'OTRO': np.int64(1)} | víctima nula: 0 | sin cod DANE: 0


Censo: 12825 filas | municipios: 125 | edades no numéricas: 0


Saber11: 297962 filas | municipios: 125 | punt_global nulo: 0
DIVIPOLA: 1122 filas | Antioquia: 125
Limpieza completa -> data/processed/*.parquet


In [5]:
import integrate
integrate.main()

Matriz municipal: 125 municipios x 26 columnas
URABA: 7811/8157 residen en Antioquia; 7811 (100.0% de los de Antioquia) cruzan con la matriz municipal
Beneficiarios: 13778/13778 (100.0%) cruzan con la matriz municipal
Saber11: 297962/297962 (100.0%) cruzan con la matriz municipal
Municipios sin dato Saber11: 0
Auditoría fuzzy censo vs DIVIPOLA: 125/125 nombres coinciden (score>=85)
Matriz de estudiantes: 8157 filas x 23 columnas | sin contexto municipal: 346
Integración completa.


## Matrices resultantes

* **Matriz municipal** (125 municipios × ~19 variables): insumo del Módulo A.
* **Matriz de estudiantes** (8.157 × 23 columnas: 15 features candidatas + contexto
  + constructoras del target): insumo del Módulo B.

El log confirma cruces del **100%** en todas las fuentes (meta del concurso: >95%).

In [6]:
m = pd.read_parquet("../data/processed/matriz_municipal.parquet")
print(f"matriz municipal: {m.shape[0]} municipios x {m.shape[1]} columnas")
m.head(3)

matriz municipal: 125 municipios x 26 columnas


,COD_DANE,NOMBRE_MUNI,POB_TOTAL,POB_15_19,POB_15_24,PCT_MUJERES_15_24,PCT_RURAL,N_MATRIC_VIVE,PCT_MUJERES_MATRIC,N_MATRIC_NACE,N_BENEF,PCT_VICTIMAS_BENEF,PROM_GLOBAL_SABER,PROM_MATEMATICAS_SABER,PROM_LECTURA_SABER,N_EVALUADOS,EVALUADOS_POR_PERIODO,BRECHA_OFICIAL_SABER,BRECHA_GENERO_SABER,NOM_MPIO,longitud,latitud,TASA_MATRIC_VIVE,TASA_MATRIC_NACE,TASA_BENEF,BRECHA_GENERO_MATRIC
0,05001,MEDELLIN,2347656,184184,410957,49.800587,1.676396,179,54.748603,714,0,NaN,250.206562,50.250501,53.581913,114842,16406.0,1.846098,-9.000536,MEDELLIN,-75.581775,6.246631,0.435569,1.737408,0.000000,4.948016
1,05002,ABEJORRAL,17599,1606,2838,45.419309,60.469345,28,67.857143,35,118,24.576271,244.700809,50.075472,51.690027,742,371.0,NaN,-4.786655,ABEJORRAL,-75.428739,5.789315,9.866103,12.332629,73.474471,22.437833
2,05004,ABRIAQUI,2159,212,388,41.237113,67.207040,0,NaN,0,25,0.000000,235.386364,48.204545,49.568182,88,44.0,NaN,9.257081,ABRIAQUI,-76.064304,6.632282,0.000000,0.000000,117.924528,NaN


In [7]:
e = pd.read_parquet("../data/processed/matriz_estudiantes.parquet")
print(f"matriz de estudiantes: {e.shape[0]} filas x {e.shape[1]} columnas")
e.head(3)

matriz de estudiantes: 8157 filas x 23 columnas


,SEXO,EDAD,ESTRATO,ESTRATO_FALTANTE,SEDE,FACULTAD,TIPO_ACEPTACION,NATURALEZA_COLE,NIVEL_PREGRADO,ANTIGUEDAD_SEMESTRES,CREDITOS_ULTIM_SEMEST_MATRIC,PCT_RURAL_MUNI,PROM_SABER_MUNI,TASA_BENEF_MUNI,VIVE_FUERA_ANTIOQUIA,PROGRAMA,COD_DANE_VIVE,NOMBRE_MUNI_VIVE,PROMEDIO_PROGRAMA,PERIODOS_PRUEBA_PROGRAMA,CREDAPROBADOS,CREDGRADO,NUMSEMESTRES
0,MASC,32.8,2,0,YARUMAL,FACULTAD DE ARTES,POR-EXAM,OFICIAL,7,7,17,72.217879,227.937500,188.158962,0,GESTION CULTURAL - YARUMAL,05854,VALDIVIA,4.68,0,118,132,7
1,FEME,24.4,3,0,CARMENVI,ESCUELA DE NUTRICION Y DIETETICA,POR-EXAM,NO OFICIAL,7,7,16,26.080029,258.495455,9.253417,0,NUTRICION Y DIETETICA - CARMEN DE VIBORAL,05440,MARINILLA,4.17,0,123,171,7
2,MASC,57.7,3,0,AMALFI,ESCUELA INTERAMERICANA DE BIBLIOTECOLOGIA,POR-EXAM,NO OFICIAL,1,1,19,1.676396,250.206562,0.000000,0,ARCHIVISTICA AMALFI,05001,MEDELLIN,9.99,0,19,145,1


In [8]:
print(open('../outputs/log_integracion.txt', encoding='utf-8').read())

Matriz municipal: 125 municipios x 26 columnas
URABA: 7811/8157 residen en Antioquia; 7811 (100.0% de los de Antioquia) cruzan con la matriz municipal
Beneficiarios: 13778/13778 (100.0%) cruzan con la matriz municipal
Saber11: 297962/297962 (100.0%) cruzan con la matriz municipal
Municipios sin dato Saber11: 0
Auditoría fuzzy censo vs DIVIPOLA: 125/125 nombres coinciden (score>=85)
Matriz de estudiantes: 8157 filas x 23 columnas | sin contexto municipal: 346
